# Code 11.7.2: A minimal PPO loop for a small language model

The third suggested code lab. It loads the reward model saved by Code 11.5.2, initializes the value model from it, and runs PPO on a few HH-RLHF-style prompts. Swap in prompts from the HH-RLHF training set (using `split_prompt` from Code 11.4.1) and increase `ITERS` for a real run; pass `beta` values from 0.01 to 0.2 for the fourth lab.

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.

This listing loads `reward_model.pt`, which Code 11.5.2 writes after training a reward model.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.
# From Code 11.2.1
import torch.nn.functional as F

# From Code 11.7.1
import torch.nn as nn

from transformers import AutoModel

# From Code 11.2.1
def generate(model, tok, prompts, max_new_tokens=24, temperature=1.0):
    """Sample one response per prompt. Returns token ids, an attention mask,
    and a mask that is 1 on response tokens up to and including the first EOS."""
    enc = tok(prompts, return_tensors="pt", padding=True)      # left-padded
    with torch.no_grad():
        seqs = model.generate(**enc, do_sample=True, top_k=0, top_p=1.0,
                              temperature=temperature, max_new_tokens=max_new_tokens,
                              pad_token_id=tok.pad_token_id)
    P = enc["input_ids"].shape[1]                              # prompt length (with padding)
    resp = seqs[:, P:]
    is_eos = resp == tok.eos_token_id
    after_eos = (is_eos.cumsum(dim=1) - is_eos.long()) > 0     # strictly after the first EOS
    resp_mask = (~after_eos).long()
    attn = torch.cat([enc["attention_mask"], resp_mask], dim=1)
    return seqs, attn, resp_mask, P

def response_logprobs(model, seqs, attn, P):
    """log pi(y_t | x, y_<t) for each response position t: shape (B, T)."""
    pos = (attn.cumsum(dim=1) - 1).clamp(min=0)                # correct positions under left padding
    logits = model(seqs, attention_mask=attn, position_ids=pos).logits
    logits = logits[:, P - 1:-1].float()                       # the logits that predict y_1 .. y_T
    logp = F.log_softmax(logits, dim=-1)
    return logp.gather(-1, seqs[:, P:].unsqueeze(-1)).squeeze(-1)

# From Code 11.5.1
class RewardModel(nn.Module):
    """A transformer backbone with a scalar head read out at the last real token."""
    def __init__(self, name):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(name)      # e.g. the SFT model without its LM head
        d = self.backbone.config.hidden_size
        self.head = nn.Linear(d, 1)
        nn.init.normal_(self.head.weight, std=1 / (d + 1) ** 0.5)
        nn.init.zeros_(self.head.bias)

    def forward(self, input_ids, attention_mask):
        h = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        last = attention_mask.sum(dim=1) - 1                 # index of the last real token (right padding)
        h_last = h[torch.arange(h.size(0)), last]
        return self.head(h_last).squeeze(-1)                 # one scalar per sequence

# From Code 11.7.1
def masked_mean(x, mask):
    return (x * mask).sum() / mask.sum()

def masked_whiten(x, mask):
    mean = masked_mean(x, mask)
    var = masked_mean((x - mean) ** 2, mask)
    return (x - mean) * torch.rsqrt(var + 1e-8)

class ValueModel(nn.Module):
    """Critic: a transformer with a scalar head at every position."""
    def __init__(self, name):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(name)
        self.head = nn.Linear(self.backbone.config.hidden_size, 1)

    def forward(self, seqs, attn, P):
        pos = (attn.cumsum(dim=1) - 1).clamp(min=0)
        h = self.backbone(input_ids=seqs, attention_mask=attn, position_ids=pos).last_hidden_state
        return self.head(h[:, P - 1:-1]).squeeze(-1)         # V(s_t) at the position that predicts y_t

def shaped_rewards(logp_old, ref_logp, score, mask, beta):
    """r_t = -beta * log(pi/pi_ref) at every token, plus the RM score at the last token."""
    log_ratio = (logp_old - ref_logp) * mask
    rewards = -beta * log_ratio
    last = mask.sum(dim=1).long() - 1                        # final response token (the EOS, if any)
    rewards[torch.arange(len(score)), last] += score
    return rewards, log_ratio

def gae(rewards, values, mask, gamma=1.0, lam=0.95):
    """Per-token advantages (whitened) and returns; masked positions contribute nothing."""
    B, T = rewards.shape
    adv = torch.zeros_like(rewards)
    last_gae = torch.zeros(B)
    for t in reversed(range(T)):
        cont = mask[:, t + 1] if t + 1 < T else torch.zeros(B)    # 0 after the last real token
        next_value = values[:, t + 1] * cont if t + 1 < T else torch.zeros(B)
        delta = rewards[:, t] + gamma * next_value - values[:, t]
        last_gae = delta + gamma * lam * cont * last_gae
        adv[:, t] = last_gae
    adv = adv * mask
    returns = adv + values
    return masked_whiten(adv, mask), returns

def ppo_loss(logp, logp_old, adv, values, values_old, returns, mask, clip=0.2, vclip=0.2):
    ratio = torch.exp(logp - logp_old)
    pg = torch.max(-adv * ratio, -adv * ratio.clamp(1 - clip, 1 + clip))
    pg_loss = masked_mean(pg, mask)
    v_clipped = values_old + (values - values_old).clamp(-vclip, vclip)
    vf = 0.5 * torch.max((values - returns) ** 2, (v_clipped - returns) ** 2)
    vf_loss = masked_mean(vf, mask)
    with torch.no_grad():
        stats = {"approx_kl": masked_mean(0.5 * (logp - logp_old) ** 2, mask).item(),
                 "clipfrac": masked_mean(((ratio - 1).abs() > clip).float(), mask).item()}
    return pg_loss, vf_loss, stats


In [ ]:
import copy, random
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0); random.seed(0)
SFT = "distilgpt2"                                   # stands in for your SFT model
beta, lr, BATCH, MINI, EPOCHS, ITERS = 0.05, 1e-5, 16, 4, 2, 20
MAX_NEW = 48                                         # token budget per reply
UNFINISHED_PENALTY = 1.0                             # a softer "EOS trick": subtracted from the score

tok = AutoTokenizer.from_pretrained(SFT)             # policy tokenizer: left padding
tok.pad_token, tok.padding_side = tok.eos_token, "left"
rm_tok = AutoTokenizer.from_pretrained(SFT)          # reward-model tokenizer: right padding
rm_tok.pad_token, rm_tok.padding_side, rm_tok.truncation_side = rm_tok.eos_token, "right", "left"
# A base model rarely emits EOS after one reply, so a newline also ends the reply
# (in HH-RLHF format the next line would start a new "Human:" turn). An SFT model would use EOS.
STOP_IDS = torch.tensor([tok.eos_token_id] + tok.encode("\n") + tok.encode("\n\n"))

def stop_mask(resp):
    """1 on response tokens up to and including the first stop token; also returns which replies ended."""
    is_stop = torch.isin(resp, STOP_IDS)
    after = (is_stop.cumsum(dim=1) - is_stop.long()) > 0
    return (~after).long(), is_stop.any(dim=1)

policy = AutoModelForCausalLM.from_pretrained(SFT)                 # actor (trained)
ref = copy.deepcopy(policy).requires_grad_(False)                  # reference (frozen)
rm = RewardModel(SFT)
rm.load_state_dict(torch.load("reward_model.pt"))                  # from Code 11.5.2
rm.requires_grad_(False)                                           # reward model (frozen)
value = ValueModel(SFT)                                            # critic (trained), initialized from the RM
value.backbone.load_state_dict(rm.backbone.state_dict())
value.head.load_state_dict(rm.head.state_dict())
for m in (policy, ref, rm, value):
    m.eval()                                                       # disable dropout everywhere
opt_pi = torch.optim.AdamW(policy.parameters(), lr=lr)
opt_v = torch.optim.AdamW(value.parameters(), lr=10 * lr)

prompts = ["\n\nHuman: How do I make a cup of tea?\n\nAssistant:",
           "\n\nHuman: Can you suggest a name for my cat?\n\nAssistant:",
           "\n\nHuman: What is a good way to learn to code?\n\nAssistant:",
           "\n\nHuman: How can I sleep better?\n\nAssistant:"]

@torch.no_grad()
def rm_score(prompts, responses):
    texts = [p + r + rm_tok.eos_token for p, r in zip(prompts, responses)]
    enc = rm_tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=512)
    return rm(enc["input_ids"], enc["attention_mask"])

for it in range(ITERS):
    # 1. Rollout: generate, then score with all four models (no gradients).
    batch = random.choices(prompts, k=BATCH)
    seqs, attn, _, P = generate(policy, tok, batch, max_new_tokens=MAX_NEW)
    mask, ended = stop_mask(seqs[:, P:])
    attn = torch.cat([attn[:, :P], mask], dim=1)                   # drop tokens after the stop
    responses = [r.strip() for r in tok.batch_decode(seqs[:, P:] * mask + tok.eos_token_id * (1 - mask),
                                                     skip_special_tokens=True)]
    mask = mask.float()
    with torch.no_grad():
        logp_old = response_logprobs(policy, seqs, attn, P)
        ref_logp = response_logprobs(ref, seqs, attn, P)
        values_old = value(seqs, attn, P)
        rm_raw = rm_score(batch, responses)
        score = rm_raw - UNFINISHED_PENALTY * (~ended).float()    # penalize replies that never end
        # 2. Per-token shaped rewards, then GAE.
        rewards, log_ratio = shaped_rewards(logp_old, ref_logp, score, mask, beta)
        adv, returns = gae(rewards, values_old, mask)
    # 3. A few epochs of clipped updates on minibatches of this rollout.
    for _ in range(EPOCHS):
        for idx in torch.randperm(BATCH).split(MINI):
            logp = response_logprobs(policy, seqs[idx], attn[idx], P)
            v = value(seqs[idx], attn[idx], P)
            pg_loss, vf_loss, stats = ppo_loss(logp, logp_old[idx], adv[idx], v, values_old[idx],
                                               returns[idx], mask[idx])
            opt_pi.zero_grad(); opt_v.zero_grad()
            (pg_loss + 0.1 * vf_loss).backward()
            torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
            torch.nn.utils.clip_grad_norm_(value.parameters(), 1.0)
            opt_pi.step(); opt_v.step()
    print(f"iter {it:2d}  score {score.mean():6.3f}  rm {rm_raw.mean():6.3f}  KL {log_ratio.sum(1).mean():6.3f}  "
          f"len {mask.sum(1).mean():5.1f}  ended {ended.float().mean():.2f}  "
          f"approx_kl {stats['approx_kl']:.4f}  clipfrac {stats['clipfrac']:.3f}")
